In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    sha2,
    concat_ws,
    col
)

import uuid

# =====================================================
# PARAMETERS
# =====================================================

dbutils.widgets.text("source_table", "")
source_table = dbutils.widgets.get("source_table")

pipeline_run_id = str(uuid.uuid4())

print(f"Processing: {source_table}")

# =====================================================
# CONFIG
# =====================================================

silver_catalog = "silver"

# source_table example:
# bronze.customer

table_name = source_table.split(".")[-1]

target_table = f"{silver_catalog}.{table_name}"

watermark_column = "migrated_at"

# =====================================================
# CREATE SCHEMA
# =====================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}")

# =====================================================
# READ SOURCE
# =====================================================

source_df = spark.table(source_table)

# =====================================================
# CREATE HASH
# =====================================================

business_columns = source_df.columns

source_df = (
    source_df
    .withColumn(
        "record_hash",
        sha2(concat_ws("||", *business_columns), 256)
    )
)

# =====================================================
# FULL LOAD
# =====================================================

if not spark.catalog.tableExists(target_table):

    print("FULL LOAD")

    final_df = (
        source_df
        .withColumn("created_at", current_timestamp())
        .withColumn("updated_at", current_timestamp())
        .withColumn("silver_ingested_at", current_timestamp())
        .withColumn("pipeline_run_id", lit(pipeline_run_id))
        .withColumn("is_active", lit(True))
    )

    (
        final_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(f"Full load completed: {target_table}")

# =====================================================
# INCREMENTAL LOAD
# =====================================================

else:

    print("INCREMENTAL LOAD")

    max_watermark = spark.sql(f"""
        SELECT MAX({watermark_column}) AS max_ts
        FROM {target_table}
    """).first()["max_ts"]

    if max_watermark is None:
        max_watermark = "1970-01-01"

    incremental_df = (
        source_df
        .filter(col(watermark_column) > lit(max_watermark))
    )

    # SERVERLESS SAFE EMPTY CHECK
    if incremental_df.limit(1).count() == 0:

        print("No new records found")

        dbutils.notebook.exit("NO_DATA")

    final_df = (
        incremental_df
        .withColumn("created_at", current_timestamp())
        .withColumn("updated_at", current_timestamp())
        .withColumn("silver_ingested_at", current_timestamp())
        .withColumn("pipeline_run_id", lit(pipeline_run_id))
        .withColumn("is_active", lit(True))
    )

    # =================================================
    # RETRY SAFE MERGE
    # =================================================

    delta_target = DeltaTable.forName(spark, target_table)

    (
        delta_target.alias("t")
        .merge(
            final_df.alias("s"),
            "t.record_hash = s.record_hash"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Incremental merge completed: {target_table}")

In [0]:

# from pyspark.sql.functions import (
#     current_timestamp,
#     lit,
#     sha2,
#     concat_ws
# )

# import uuid

# # =========================================================
# # CONFIG READ
# # =========================================================

# configs = spark.sql("""
#     SELECT DISTINCT source_table
#     FROM configs.migration_config
#     WHERE enabled = true
# """).collect()

# bronze_schema = "bronze"
# silver_schema = "silver"

# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

# # =========================================================
# # FUNCTION FOR SINGLE TABLE PIPELINE
# # =========================================================

# def process_table(source_table_full):

#     source_table = source_table_full
#     target_table = f"silver.{source_table}"

#     print(f"\n========== Processing {source_table} ==========")

#     table_exists = spark.catalog.tableExists(target_table)

#     # =====================================================
#     # FULL LOAD
#     # =====================================================

#     if not table_exists:

#         print("FULL LOAD")

#         df = spark.table(source_table)

#         df = df.withColumn(
#             "record_hash",
#             sha2(concat_ws("||", *df.columns), 256)
#         )

#         df = df.withColumn("created_at", current_timestamp()) \
#                .withColumn("updated_at", current_timestamp()) \
#                .withColumn("is_active", lit(True)) \
#                .withColumn("silver_ingested_at", current_timestamp())

#         df.write.format("delta") \
#             .mode("overwrite") \
#             .saveAsTable(target_table)

#         print(f"Full load completed: {source_table}")

#     # =====================================================
#     # INCREMENTAL LOAD
#     # =====================================================

#     else:

#         print("INCREMENTAL LOAD")

#         last_ts = spark.sql(f"""
#             SELECT MAX(created_at) AS last_ts
#             FROM {target_table}
#         """).collect()[0]["last_ts"]

#         if last_ts is None:
#             last_ts = "1970-01-01 00:00:00"

#         df = spark.sql(f"""
#             SELECT *
#             FROM {source_table}
#             WHERE migrated_at > TIMESTAMP('{last_ts}')
#         """)

#         if df.rdd.isEmpty():
#             print(f"No new data: {source_table}")
#             return

#         df = df.withColumn(
#             "record_hash",
#             sha2(concat_ws("||", *df.columns), 256)
#         )

#         df = df.withColumn("created_at", current_timestamp()) \
#                .withColumn("updated_at", current_timestamp()) \
#                .withColumn("is_active", lit(True)) \
#                .withColumn("silver_ingested_at", current_timestamp())

#         df.write.format("delta") \
#             .mode("append") \
#             .saveAsTable(target_table)

#         print(f"Incremental load completed: {source_table}")

# # =========================================================
# # PARALLEL EXECUTION
# # =========================================================

# rdd = spark.sparkContext.parallelize(configs)

# rdd.foreach(lambda x: process_table(x["source_table"]))